## Data analysis & Visualization - Day 6 - LAB 6

In [ ]:
import pandas as pd

**Task 1: The mean and the median**

In [ ]:
listings_cl_raw = pd.read_csv('data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])

listings = listings_cl_raw.copy()

print(listings.shape)
print(listings['price'].dtype, listings['last_review'].dtypes)

In [ ]:
listings[['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']].dtypes

In [ ]:
# Note to oneself: One column is called a series

reviews_small = pd.Series([3, 5, 8, 10, 12]) # This range can be said to be evenly spread.

print("Mean:", reviews_small.mean(), "\nMedian",reviews_small.median()) # In this case, when the range is evenly spread (without outliars), mean and median gives the same or similar output.

In [ ]:
reviews_with_extreme = pd.Series([3, 5, 8, 10, 12, 1293]) # Here we have a clear outliar at the max of this range.

print("Mean:", round(reviews_with_extreme.mean(), 1), "\nMedian:", reviews_with_extreme.median()) # With an outliar at max, we clearly see how this extreme pulls the average (mean) away from the median.

In [ ]:
n_reviews = listings['number_of_reviews'] # Using a column from the DataFrame.

print("Mean", n_reviews.mean().round(1))
print("Median", n_reviews.median())
#Conclusively, the median of 9 reviews is more useful since listings with many reviews are concentrated in the upper part of the range.

___

**Task 2: Percentiles and the spread**

In [ ]:
n_reviews.describe() # Display descriptive statistics.

In [ ]:
# Calculate selected percentiles.
n_reviews.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
# Calculate the quartiles and interquartile range.
q1 = n_reviews.quantile(0.25)
q3 = n_reviews.quantile(0.75)
iqr = q3 - q1

print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr) # The IQR shows the spread in the Q2 range.

In [ ]:
# Calculating the procentage of "missing" reviews.
print(((n_reviews == 0).mean() * 100).round(2))

___

**Task 3: The fences and the outliers**

In [ ]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print('Lower fence:', lower)
print('Upper fence:', upper)

In [ ]:
listings[listings['number_of_reviews'] < lower]

The lower fence is -38.5. This demonstrates that an IQR fence is not guaranteed to be a meaningful real-world value.

In [ ]:
reviews_outliers = listings[
    listings['number_of_reviews'] > upper
]

In [ ]:
reviews_outliers.shape[0]

In [ ]:
print((n_reviews > upper).sum())

In [ ]:
print(((n_reviews > upper).mean() * 100).round(1))

In [ ]:
reviews_outliers['room_type'].value_counts()

In [ ]:
outlier_median_price = reviews_outliers['price'].median()

other_median_price = listings.loc[listings['number_of_reviews'] <= upper, 'price'].median()

print('Outlier median price:', outlier_median_price)
print('Other listings median price:', other_median_price)

**The lower fence has a negative value, which makes lower review outliers impossible. There are 582 listings above the upper fence of 69.5 reviews. Most are entire homes/apartments, but that is also the largest room type in the full dataset. The median price is lower among the review outliers, but this comparison does not establish that price caused the difference in the number of reviews.**

___

**Task 4 - Mark them, and compare**

In [ ]:
listings['reviews_outlier'] = (
    listings['number_of_reviews'] > upper
)

listings['reviews_outlier'].value_counts()

In [ ]:
reviews_all = listings['number_of_reviews']

reviews_without = listings.loc[~listings['reviews_outlier'], 'number_of_reviews']

In [ ]:
statistics = ['count', 'mean', 'median', 'std']

comparison = pd.DataFrame({
    'all': reviews_all.agg(statistics),
    'without_outliers': reviews_without.agg(statistics)
})

comparison.round(1)

In [ ]:
comparison['change_percent'] = (
    (
        comparison['without_outliers']
        - comparison['all']
    )
    / comparison['all']
    * 100
)

comparison.round(1)

**Removing the marked review outliers reduces the mean by about 64% and the standard deviation by about 80%, but the median falls by only about 22%. The extreme review counts strongly affect the mean and spread, while the median is more resistant. The outliers remain marked in the original table rather than being deleted.**

___

**Task 5 - Describe the groups**

In [ ]:
reviews_by_room = (
    listings
    .groupby('room_type')['number_of_reviews']
    .describe()
    .round(0)
)

reviews_by_room

**Shared rooms have the highest median number of reviews, with 82, based on 24 listings. I would not draw a conclusion about hotel rooms because that group contains only three listings. Group statistics should be interpreted together with their counts.**

___

**Task 6 - Correlation**

In [ ]:
correlation_columns = [
    'number_of_reviews',
    'price',
    'accommodates',
    'bedrooms',
    'availability_365',
    'review_scores_rating',
    'minimum_nights',
    'days_since_review'
]

review_correlations = (
    listings[correlation_columns]
    .corr()['number_of_reviews']
    .round(2)
)

review_correlations

In [ ]:
corr_all = listings[
    'number_of_reviews'
].corr(
    listings['availability_365']
)

print(round(corr_all, 2))

In [ ]:
without_outliers = listings[
    ~listings['reviews_outlier']
]

corr_without = without_outliers[
    'number_of_reviews'
].corr(
    without_outliers['availability_365']
)

print(round(corr_without, 2))

In [ ]:
print('With all listings:', round(corr_all, 2))
print('Without review outliers:', round(corr_without, 2))

**All relationships with number_of_reviews are weak. The strongest is the negative relationship with days_since_review (-0.19), and the weakest is with review_scores_rating (-0.04). The correlation with availability_365 falls from 0.18 to 0.00 when review outliers are excluded, showing that the original relationship is driven by the unusually highly reviewed listings. Correlation does not show causation.**

___